# NUTS4

## Notes

### EuroStat

https://ec.europa.eu/eurostat/web/gisco/geodata/reference-data/administrative-units-statistical-units/nuts

Consider to retrive data from API <br>
https://ec.europa.eu/eurostat/web/main/data/web-services

## Load

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# %load import.py
#
from IPython.display import IFrame

# 
import numpy as np
import pandas as pd

#
from sentinelsat import SentinelAPI, read_geojson, geojson_to_wkt
from datetime import date

#
import rioxarray
import geopandas as gpd
import rasterio as rio

#
from matplotlib import pyplot
from rasterio.plot import show

#
from sqlalchemy import create_engine # query PostGIS
from sqlalchemy import inspect

#
import osmnx as ox

#
from shapely.geometry import Polygon, box, MultiPolygon
import shapely.ops as so

#
import json

#
import os
from pathlib import Path
import fnmatch
import glob

#
from osgeo import gdal

#
import random

#
import shutil

#
from tqdm import tqdm
import time

#
import folium
from folium import plugins

## PostGIS

### Study Area

In [ ]:
# sa_fil = "italy_center_south.geojson"
sa_fil = "naples_metropolytan.geojson"

In [ ]:
sa = gpd.read_file(sa_fil)

In [ ]:
sa.plot()

### Get list of NUTS4 within Study Area (sa)

#### Source: LandSupport DEV
 - reference code is in [LTM_ADVANCED_EU.ipynb](http://192.168.30.11:8888/notebooks/release/work/repo/middleware/api/PPProcessor/jupyter/management/LTM_ADVANCED_EU.ipynb) <br>
 - reference lib in in [psql_lib.ipynb](http://192.168.30.11:8888/notebooks/release/work/repo/middleware/api/PPProcessor/jupyter/lib/psql_lib.ipynb) (which I should download locally as `psql_lib.py`)

##### Using LandSupport lib | DEPRECATED

##### Using GeoPandas | easiest way

In [ ]:
import os
db_connection_url = os.environ["LANDSUPPORT_URL"]
con = create_engine(db_connection_url)  

In [ ]:
sa.geometry

In [ ]:
sql = """
SELECT nuts_name,geom
	FROM public.nuts_4_2013 a
	WHERE ST_Intersects( a.geom, ST_GeomFromText('""" + str(sa.geometry[0]) + """',4326)
					 )
"""
print(sql)

In [ ]:
cities_gdf = gpd.read_postgis(sql, con)

In [ ]:
cities_gdf

In [ ]:
cities_gdf["nuts_name"][0]

##### Check geometries - lots of MultiPolygons!

In [ ]:
naisc = gpd.read_file("nuts4_sa_nap-ischia.geojson")
naisc

In [ ]:
naisc_e = naisc.explode(index_parts=False)
naisc_e

In [ ]:
geom_mp_na = MultiPolygon(naisc_e.geometry.values)
geom_mp_na

In [ ]:
naisc_mp = gpd.GeoDataFrame({'id':[0],'geometry':[geom_mp_na]}, crs=4326)
naisc_mp

In [ ]:
naisc_mp.to_file("na-isc-multipolygon.geojson")

In [ ]:
naisc_mp = gpd.read_file("na-isc-multipolygon.geojson")

In [ ]:
naisc_mp.explode(index_parts=False)

##### Process geodataframe to get clean Polygons

The list of geometries got from LandSupport DB has a lot of MultiPolygons where it is actually a Polygon.<br>
Here I have to convert MultiPolygons into Polygons, if possibile.

In [ ]:
#...ToDo

##### Search for duplicated nuts4

It may happen that the same city has two overlapping geometries (see Procida).<br>
In that case, I need a preliminary check to avoid duplicate nuts4.

In [ ]:
#...ToDo

##### Save vector data

In [ ]:
cities_gdf.to_file("nuts4_sa_napoli.geojson")

##### Plot geodataframe - Folium

https://www.linkedin.com/pulse/visualize-dem-interactive-map-chonghua-yin/?trk=related_artice_Visualize%20DEM%20in%20An%20Interactive%20Map_article-card_title

In [ ]:
cities_gdf = gpd.read_file("nuts4_sa_napoli.geojson")

In [ ]:
nuts4_idx = 0

In [ ]:
print(cities_gdf.geometry[nuts4_idx])

In [ ]:
m = folium.Map([40, 14], zoom_start=7, tiles='cartodbpositron')
folium.GeoJson('naples_metropolytan.geojson').add_to(m)
folium.LatLngPopup().add_to(m)
#m.fit_bounds([[xn,yn],[xx,yx]])
m

### Get list of NUTS4 within Products

#### To be done...